<a href="https://colab.research.google.com/github/LohanVaddepally/Store-Analytics-Dashboard/blob/main/Store_Analytics_Dasboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===============================================
# Store Sales Forecast & Recommendations
# ===============================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import plotly.graph_objects as go
from ipywidgets import widgets, interactive

# ======================
# DATA
# ======================
data = {
    "store": ["store A", "store B", "store C", "store D"],
    "sales_last_3_months": [
        [45000, 47000, 50000],
        [21000, 20000, 20000],
        [68000, 69000, 70000],
        [18000, 19000, 18000]
    ],
    "inventory": [10000, 12000, 15000, 9000],
    "returns": [3000, 4000, 1000, 5000]
}

df = pd.DataFrame(data)

# ======================
# KPI CALCULATIONS
# ======================
df["avg_sales"] = df["sales_last_3_months"].apply(np.mean)
df["return_rate"] = df["returns"] / df["avg_sales"]
df["inventory_turnover"] = df["avg_sales"] / df["inventory"]

# Simple linear regression to forecast next month
def forecast_sales(history):
    X = np.array([0,1,2]).reshape(-1,1)
    y = np.array(history)
    model = LinearRegression()
    model.fit(X,y)
    return float(model.predict(np.array([[3]])))

df["forecast_sales"] = df["sales_last_3_months"].apply(forecast_sales)

# Recommendation logic
def recommendation(row):
    if row["forecast_sales"] > row["inventory"] * 1.2:
        return "Order More Stock"
    elif row["return_rate"] > 0.15:
        return "Investigate Returns"
    elif row["inventory_turnover"] < 2:
        return "Reduce Inventory"
    else:
        return "Maintain"

df["action"] = df.apply(recommendation, axis=1)

# ======================
# INTERACTIVE DASHBOARD
# ======================
def display_store(store_choice):
    store_data = df[df["store"] == store_choice].iloc[0]

    # Display key metrics
    print(f"Store: {store_choice}")
    print(f"Average Sales: ${store_data['avg_sales']:.0f}")
    print(f"Return Rate: {store_data['return_rate']*100:.1f}%")
    print(f"Inventory Turnover: {store_data['inventory_turnover']:.2f}")
    print(f"Forecasted Sales: ${store_data['forecast_sales']:.0f}")
    print(f"Recommended Action: {store_data['action']}\n")

    # Plot sales trend
    months = ["M1", "M2", "M3", "Forecast M4"]
    sales_values = store_data["sales_last_3_months"] + [store_data["forecast_sales"]]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=months, y=sales_values, mode="lines+markers", name="Sales"))
    fig.add_vline(x=2.5, line_dash="dash", line_color="red", annotation_text="Forecast")
    fig.update_layout(
        title=f"{store_choice} Sales Trend",
        yaxis_title="Sales ($)",
        xaxis_title="Month"
    )
    fig.show()

store_dropdown = widgets.Dropdown(
    options=df["store"].tolist(),
    description='Select Store:'
)

interactive(display_store, store_choice=store_dropdown)

/tmp/ipykernel_23970/1507939190.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(model.predict(np.array([[3]])))
/tmp/ipykernel_23970/1507939190.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(model.predict(np.array([[3]])))
/tmp/ipykernel_23970/1507939190.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(model.predict(np.array([[3]])))
/tmp/ipykernel_23970/1507939190.py:41: DeprecationWarning: Conversion of an arr

interactive(children=(Dropdown(description='Select Store:', options=('store A', 'store B', 'store C', 'store D…